In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from viz_config import VizConfig

# ==========================================
# 0. Global configuration and style setup
# ==========================================
# Import the shared colour/style configuration so every figure meets publication standards
VizConfig.set_style()

# Output directory - generated PDFs land here
OUTPUT_DIR = "Refined_Results_v4"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 1. Data loading and preprocessing
# ==========================================
csv_path = os.path.join(OUTPUT_DIR, "final_summary_r2.csv")

# If the data file is missing, generate synthetic data to exercise the plotting code
if not os.path.exists(csv_path):
    print(f"Note: {csv_path} missing - using synthetic data for the demo.")
    # Synthetic data: noise levels from 0% to 200%
    # R2 Stay high until ~45%, then drop - illustrates the robustness boundary
    x_raw = np.array([0, 0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
    y_pysr = np.array([0.99, 0.99, 0.98, 0.97, 0.96, 0.96, 0.96, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.74, 0.72, 0.70, 0.65, 0.59, 0.51])
    df = pd.DataFrame({'Noise_Ratio': x_raw, 'R2_Direct_PySR': y_pysr})
else:
    # Load real experimental data
    df = pd.read_csv(csv_path)
    x_raw = df['Noise_Ratio'].values
    y_pysr = df['R2_Direct_PySR'].values

# ==========================================
# 2. Plot initialisation
# ==========================================
fig, ax = plt.subplots(figsize=(10, 7))

# Define"breakdown point" (Critical Breakdown Point)  X position
# Empirically, performance drops sharply around 47.5%
cliff_x = 0.475  
max_x = max(x_raw) if max(x_raw) > 2.0 else 2.0 

# ==========================================
# 3. Core plot: background zones and data curves
# ==========================================

# A. Plot the background zones
# Use different background colours to split the model's"robust zone" and "failure zone"
# Left: robust zone (light green, COLOR_SUCCESS)
ax.axvspan(xmin=-0.05, xmax=cliff_x, color=VizConfig.COLOR_SUCCESS, alpha=0.08, zorder=0) 
# Right: failure zone (light red, COLOR_HIGHLIGHT)
ax.axvspan(xmin=cliff_x, xmax=max_x+0.1, color=VizConfig.COLOR_HIGHLIGHT, alpha=0.08, zorder=0)

# B. Plot the breakdown threshold line
# Vertical dashed line at the breakdown point as a visual divider
ax.axvline(x=cliff_x, color=VizConfig.COLOR_SECONDARY, linestyle='--', linewidth=1.5, zorder=1)

# C. Plot the performance-degradation curve
# Shows R^2 vs. noise level
# Use COLOR_MAIN for the line and markers
ax.plot(x_raw, y_pysr, marker='o', markersize=8, linewidth=2.5, 
        color=VizConfig.COLOR_MAIN, label='Direct Symbolic Regression (Baseline)', zorder=2,
        markeredgecolor='white', markeredgewidth=1.5)

# ==========================================
# 4. Annotations and styling
# ==========================================

# A. Add zone labels
# Place labels at the bottom to avoid covering the data
text_y_pos = 0.05 
# "Robust Zone" annotation - green bold
ax.text(x=cliff_x/2, y=text_y_pos, s="Robust Zone", 
        color=VizConfig.COLOR_SUCCESS, fontsize=VizConfig.LABEL_SIZE, fontweight='bold', ha='center', va='bottom')
# "Failure Zone" annotation - red bold
ax.text(x=cliff_x + (max_x - cliff_x)/2, y=text_y_pos, s="Failure Zone", 
        color=VizConfig.COLOR_HIGHLIGHT, fontsize=VizConfig.LABEL_SIZE, fontweight='bold', ha='center', va='bottom')

# B. Annotate the critical breakdown point
# Annotate mid-way along the divider with a white bbox for legibility
ax.text(x=cliff_x, y=0.5, s="Critical Breakdown Point", 
        color=VizConfig.COLOR_AXIS, fontsize=12, ha='center', va='bottom', fontweight='bold',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

# C. XAxis formatting
# X ticks every 20% (0.2), formatted as a percent string (e.g. "20%")
x_ticks = np.arange(0, max_x + 0.1, 0.2)
x_tick_labels = [f"{int(val*100)}%" for val in x_ticks]

ax.set_xticks(x_ticks)
ax.set_xticklabels(x_tick_labels, rotation=0, fontsize=VizConfig.TICK_SIZE)
ax.tick_params(axis='y', labelsize=VizConfig.TICK_SIZE)

# ==========================================
# 5. Title and legend
# ==========================================
# Axis labels and title (LaTeX supported, e.g. $R^2$)
ax.set_xlabel("Noise Level (%)", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax.set_ylabel(r"Coefficient of Determination ($R^2$)", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
ax.set_title("Model Performance Degradation Curve", fontsize=VizConfig.TITLE_SIZE, pad=20, fontweight='bold')

# Legend (top-right)
ax.legend(fontsize=VizConfig.LEGEND_SIZE, loc='upper right', frameon=True, facecolor='white', framealpha=1) 

# Light dashed grid to help read values
ax.grid(True, linestyle='--', alpha=0.4, color=VizConfig.COLOR_GRID, zorder=1)

# Set axis limits
ax.set_xlim(0, 2.05) 
ax.set_ylim(-0.05, 1.1) 

# ==========================================
# 6. Save and show
# ==========================================
plt.tight_layout()
output_path = os.path.join(OUTPUT_DIR, "4_refined.pdf")
# Save as a 300-DPI PDF
plt.savefig(output_path, dpi=VizConfig.DPI)

print(f"Figure saved to: {output_path}")
plt.show()